# 1 — Setup and Configuration
**Image Enhancement as Preprocessing for Industrial Anomaly Detection Under Challenging Illumination.** Team-approved **512×512 RGB + exact foreground-coverage masks + layer2-derived 64×64 PaDiM grid**, applied uniformly to all eight MVTec AD 2 categories and four conditions. This notebook relies on accepted dataset qualification and leaves original files untouched.
Use a fresh Colab Python 3 runtime and **Runtime → Run all**. Default `small` reuses the exact archived pilot identities/order (25 normal references; 10 good + 10 bad evaluations per category), without resampling. CPU works; GPU is optional. ImageNet weights may download once. Each run creates a new timestamped output folder and begins incomplete.
This is a **new protocol**, separate from archived 256/NEAREST results. Team accepted region merging, occupancy-area inflation, and square aspect distortion. Batch 3 found substantial fabric image-AUROC regressions; no universal accuracy improvement is claimed. Sam reviews the fresh small evidence before medium.


In [ ]:
# Configure workload before dataset reading, model loading, or feature extraction.
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
from datetime import datetime, timezone
import csv
import json
import random
import time
import platform
import hashlib
import io
import sys
try:
    import resource
except ImportError:
    resource = None  # CPU RSS measurement is unavailable on Windows; never invent it.
import numpy as np
import scipy
from scipy.ndimage import label as connectedComponents, gaussian_filter
import sklearn
from sklearn.metrics import roc_auc_score, roc_curve, auc
import cv2
import PIL
from PIL import Image
import torch
import torchvision
from torchvision.models import resnet18, ResNet18_Weights
from torch.nn import functional as F

workloadSize = 'small'  # 'small', 'medium', 'large'
randomSeed = 7367
allCategories = ['can', 'fabric', 'fruit_jelly', 'rice', 'sheet_metal', 'vial', 'wallplugs', 'walnuts']
selectedCategories = list(allCategories)  # Or an explicit subset, e.g. ['can', 'wallplugs'].
if (not selectedCategories or len(selectedCategories) != len(set(selectedCategories))
        or not set(selectedCategories).issubset(allCategories)):
    raise ValueError('selectedCategories must be a nonempty, unique subset of the eight MVTec AD 2 categories.')
workloadLimits = {'small': (25, 20), 'medium': (250, 100), 'large': (None, None)}
if workloadSize not in workloadLimits:
    raise ValueError("workloadSize must be 'small', 'medium', or 'large'.")
referenceCap, evaluationCap = workloadLimits[workloadSize]
datasetRoot = Path('/content/drive/MyDrive/ColabData/CS7367_Vision/mvtec_ad_2')
outputRoot = Path('/content/drive/MyDrive/ColabData/CS7367_Vision/output/experiments')
archivedPilotRoot = outputRoot / 'small' / '20261115T000643_647170Z'
archivedManifestSha256 = '0588df4ceceabe99b7d4fab5a6798d13df467b02863d9da42517a1edd907c7c0'
categories = selectedCategories
conditions = ['baseline', 'gamma', 'histogram_equalization', 'clahe']
imageSize = 512
featureGrid = None  # Derived from the actual layer2 tensor after backbone loading.
featureDimension = 32
batchSize = 5
covarianceRegularization = 0.01
enhancementParameters = {'gamma': 0.7, 'claheClipLimit': 2.0, 'claheTileGrid': [8, 8],
                         'channel': 'CIELAB L after RGB bilinear resize; baseline RGB unchanged'}
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
random.seed(randomSeed)
np.random.seed(randomSeed)
torch.manual_seed(randomSeed)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(randomSeed)
if device.type == 'cuda':
    torch.cuda.reset_peak_memory_stats(device)
torch.backends.cudnn.benchmark = False
torch.use_deterministic_algorithms(True)
runStarted = time.perf_counter()
runTimestamp = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S_%fZ')
runRoot = outputRoot / workloadSize / runTimestamp
if runRoot.resolve().is_relative_to(datasetRoot.resolve()):
    raise ValueError('Outputs must be outside the original dataset.')
runRoot.mkdir(parents=True, exist_ok=False)  # Never overwrite an existing run.
metricFields = ['dataset', 'category', 'condition', 'workloadMode', 'referenceCount',
                'evaluationCount', 'normalEvaluationCount', 'anomalyEvaluationCount',
                'metricName', 'metricValue', 'metricStatus', 'runtimeSeconds']
metricRows = []
manifestRows = []
selectedData = {}
summary = dict(implementationVersion='research-batch4', protocolVersion='512-coverage-layer2-v1', purpose='exploratory category screening; no final category or winning-method claim',
               stageTimings=[], maskDiagnostics=[], maskAuditOutcomes=[], maskAuditComplete=False, fileManifest=[],
               maskResamplingPolicy='Exact foreground occupancy over edge-aligned source pixel-cell footprints; 512x512 binary output, uniform across categories/conditions',
               comparisonPolicy='Distinct 512/coverage/grid64 protocol; archived 256/NEAREST/grid32 values must not be combined into this series. Compare descriptively with identical identities.',
               identityPolicy='small: frozen archived manifest and exact order; medium/large: unchanged seed-stratified selection',
               memoryUsage={'cpuProcessLifetimePeakRssBytes': None, 'gpuPeakAllocatedBytes': None, 'scope': 'process-lifetime CPU peak via resource where available; GPU peak since setup'},
               status='incomplete', workloadMode=workloadSize, configuredCaps={'reference': referenceCap, 'evaluation': evaluationCap},
               seed=randomSeed, utcTimestamp=runTimestamp, selectedCategories=categories,
               splitPolicy='Fit train/good only; evaluate test_public/good and test_public/bad only; validation and private tests excluded.',
               enhancementParameters=enhancementParameters, conditions=conditions,
               model={'name': 'PaDiM', 'backbone': 'ResNet18', 'weights': 'IMAGENET1K_V1', 'layers': ['layer1', 'layer2', 'layer3'],
                      'inputSize': imageSize, 'featureGrid': featureGrid, 'featureDimension': featureDimension,
                      'covarianceRegularization': covarianceRegularization, 'batchSize': batchSize, 'smoothingSigma': 4.0},
               metricPolicy='Image AUROC on public labels; normalized AU-PRO0.05 at 512x512 coverage masks, 8-connected regions, all selected public masks required.',
               environment={'python': platform.python_version(), 'torch': torch.__version__, 'torchvision': torchvision.__version__,
                            'numpy': np.__version__, 'scipy': scipy.__version__, 'sklearn': sklearn.__version__,
                            'opencv': cv2.__version__, 'pillow': PIL.__version__, 'device': str(device)},
               configuredDatasetRoot=str(datasetRoot), outputPath=str(runRoot), actualCounts={}, metricAvailability=[],
               limitations=['Small/medium AUROC describes the selected balanced sample, not full-set AUROC.',
                            'Coverage preserves positive occupancy, can merge regions and inflate area; fixed square resizing distorts aspect ratio. These accepted semantics are not native-resolution certification.',
                            'No private-test illumination-shift metrics are inferred; public evaluation cannot replace private-server evaluation.',
                            'Batch 3 fabric AUROC regressed; 512 does not establish a universal accuracy gain. Review the new small pilot before medium.',
                            'Fixed 32-dimensional features and layer2-derived spatial alignment are a bounded PaDiM configuration, not a claimed reproduction of published scores.'],
               errors=[], totalRuntimeSeconds=0)

stageStarts = {}

def logStage(event, stage, category='all', condition='all', processed=None, total=None):
    nowUtc = datetime.now(timezone.utc).strftime('%Y-%m-%dT%H:%M:%S.%fZ')
    nowCounter = time.perf_counter()
    key = (stage, category, condition)
    if event == 'START':
        timing = dict(stage=stage, category=category, condition=condition, startUtc=nowUtc,
                      endUtc=None, elapsedSeconds=None, status='incomplete')
        stageStarts[key] = (nowCounter, timing)
        summary['stageTimings'].append(timing)
    elapsed = nowCounter - stageStarts[key][0] if key in stageStarts else nowCounter - runStarted
    if event == 'COMPLETE':
        _, timing = stageStarts.pop(key)
        timing.update(endUtc=nowUtc, elapsedSeconds=elapsed, status='complete')
    progress = '' if processed is None else f' | {processed}/{total} images'
    context = f' | mode={workloadSize} | run={runTimestamp} | device={device}' if stage == 'notebook' else ''
    print(f'[{nowUtc}] {event} | {category} | {condition} | {stage}{progress} | elapsed {elapsed:.1f}s{context}', flush=True)

def saveEvidence():
    logStage('START', 'evidence writing')
    summary['totalRuntimeSeconds'] = time.perf_counter() - runStarted
    with (runRoot / 'metrics.csv').open('w', newline='', encoding='utf-8') as handle:
        writer = csv.DictWriter(handle, fieldnames=metricFields)
        writer.writeheader()
        writer.writerows(metricRows)
    temporaryPath = runRoot / 'experiment_summary.json.tmp'
    temporaryPath.write_text(json.dumps(summary, indent=2, allow_nan=False), encoding='utf-8')
    temporaryPath.replace(runRoot / 'experiment_summary.json')
    logStage('COMPLETE', 'evidence writing')
    # Persist the observed end timing after the evidence has actually been written.
    temporaryPath.write_text(json.dumps(summary, indent=2, allow_nan=False), encoding='utf-8')
    temporaryPath.replace(runRoot / 'experiment_summary.json')

logStage('START', 'notebook')
saveEvidence()


# 2 — Dataset Selection and Sampling
Fit **train/good only**; validation is unused. Evaluate only **test_public/good** and **test_public/bad**. The default small workload loads the frozen 256-pilot manifest, verifies its SHA-256, split/role/label consistency, all selected files, counts, identities, and exact row order. A missing/changed archive stops execution; there is no fallback resampling. The new protocol reuses identities, not old metric values.
Medium/large retain the original deterministic selector and caps. All four conditions reuse the same selected lists; roles are disjoint. The explicit category subset selector preserves archived small-manifest order for that subset.


In [ ]:
# Select identities once, before enhancement. No file contents are altered.
def selectImages(groups, cap, seed):
    rng = random.Random(str(seed))
    pools = {group: sorted(paths) for group, paths in sorted(groups.items()) if paths}
    groupNames = sorted(pools)
    rng.shuffle(groupNames)
    for pool in pools.values():
        rng.shuffle(pool)
    selected = []
    total = sum(map(len, pools.values())) if cap is None else cap
    while len(selected) < total and any(pools.values()):
        for group in groupNames:
            if pools[group] and len(selected) < total:
                selected.append((group, pools[group].pop()))
    return selected

def validateManifest(manifest):
    if len({r['relativeImagePath'] for r in manifest}) != len(manifest):
        raise ValueError('Duplicate/leaking manifest identities.')
    for row in manifest:
        relative = Path(row['relativeImagePath'])
        if relative.is_absolute() or '..' in relative.parts:
            raise ValueError('Manifest path must be relative to the dataset.')
        if row['role'] == 'reference' and row['group'] == 'good' and row['label'] == '0':
            expected = Path(row['category']) / 'train/good'
        elif row['role'] == 'evaluation' and row['group'] in ['good', 'bad'] and row['label'] == str(int(row['group'] == 'bad')):
            expected = Path(row['category']) / 'test_public' / row['group']
        else:
            raise ValueError('Manifest role/group/label mismatch.')
        if relative.parent != expected:
            raise ValueError('Manifest split disagrees with its role; refuse evaluation-to-reference leakage.')

logStage('START', 'dataset discovery and sampling')
try:
    if not datasetRoot.is_dir():
        raise FileNotFoundError(datasetRoot)
    if not any((datasetRoot / c).is_dir() for c in categories):
        candidates = sorted({p.parent for c in categories for p in datasetRoot.rglob(c) if p.is_dir()})
        if len(candidates) != 1:
            raise ValueError('Cannot uniquely locate the MVTec category root.')
        datasetRoot = candidates[0]
    summary['accessibleDatasetRoot'] = str(datasetRoot)
    archivedRows = None
    if workloadSize == 'small':
        archivedBytes = (archivedPilotRoot / 'sample_manifest.csv').read_bytes()
        digest = hashlib.sha256(archivedBytes).hexdigest()
        if digest != archivedManifestSha256:
            raise ValueError('Archived small manifest is missing/changed; refusing to resample identities.')
        archivedRows = list(csv.DictReader(io.StringIO(archivedBytes.decode('utf-8-sig'))))
        validateManifest(archivedRows)
        if {r['category'] for r in archivedRows} != set(allCategories):
            raise ValueError('Archived manifest must contain the eight approved categories.')
        manifestRows = [dict(r) for r in archivedRows if r['category'] in categories]
        summary['archivedIdentityManifest'] = {'path': str(archivedPilotRoot / 'sample_manifest.csv'), 'sha256': digest}
        summary['archivedProtocol'] = '256x256 RGB / NEAREST masks / grid32; identity source only; non-combinable metric series'
    for category in categories:
        categoryRoot = datasetRoot / category
        if archivedRows is not None:
            references = [(r['group'], datasetRoot / r['relativeImagePath']) for r in archivedRows if r['category'] == category and r['role'] == 'reference']
            evaluations = [(r['group'], datasetRoot / r['relativeImagePath']) for r in archivedRows if r['category'] == category and r['role'] == 'evaluation']
            if len(references) != 25 or len(evaluations) != 20 or sum(g == 'bad' for g, _ in evaluations) != 10:
                raise ValueError(f'{category}: frozen small workload must be exactly 25 references, 10 good and 10 bad evaluations.')
            for _, path in references + evaluations:
                if not path.is_file():
                    raise FileNotFoundError(f'Archived identity unavailable: {path}')
        else:
            groups = {}
            for group in ['train/good', 'test_public/good', 'test_public/bad']:
                folder = categoryRoot / group
                groups[group] = sorted(p for p in folder.glob('*') if p.is_file() and p.suffix.lower() in ['.png', '.jpg', '.jpeg', '.tif', '.tiff', '.bmp'])
                if not groups[group]:
                    summary['limitations'].append(f'{category}: unavailable or empty {group}.')
            references = selectImages({'good': groups['train/good']}, referenceCap, f'{randomSeed}|{category}|reference')
            evaluations = selectImages({'good': groups['test_public/good'], 'bad': groups['test_public/bad']}, evaluationCap, f'{randomSeed}|{category}|evaluation')
        if {p.resolve() for _, p in references} & {p.resolve() for _, p in evaluations}:
            raise ValueError(f'{category}: reference/evaluation identity leakage.')
        selectedData[category] = {'reference': references, 'evaluation': evaluations}
        counts = dict(reference=len(references), evaluation=len(evaluations),
                      normalEvaluation=sum(g == 'good' for g, _ in evaluations), anomalyEvaluation=sum(g == 'bad' for g, _ in evaluations))
        summary['actualCounts'][category] = counts
        for role, records in ([] if archivedRows is not None else selectedData[category].items()):
            for group, imagePath in records:
                manifestRows.append(dict(category=category, role=role, label=int(group == 'bad'), group=group,
                                         relativeImagePath=imagePath.relative_to(datasetRoot).as_posix(), workloadMode=workloadSize))
        logStage('INFO', f'selected reference={len(references)}, evaluation={len(evaluations)} (good={counts["normalEvaluation"]}, bad={counts["anomalyEvaluation"]})', category)
    with (runRoot / 'sample_manifest.csv').open('w', newline='', encoding='utf-8') as handle:
        writer = csv.DictWriter(handle, fieldnames=['category', 'role', 'label', 'group', 'relativeImagePath', 'workloadMode'])
        writer.writeheader()
        writer.writerows(manifestRows)
    with (runRoot / 'sample_manifest.csv').open(newline='', encoding='utf-8') as handle:
        writtenRows = list(csv.DictReader(handle))
    if archivedRows is not None and writtenRows != manifestRows:
        raise ValueError('Written small manifest differs from the exact archived identities/order.')
    summary['fileManifest'] = writtenRows
    summary['archivedManifestComparison'] = {'matched': True, 'selectedRows': len(writtenRows)} if archivedRows is not None else {'matched': None, 'reason': 'not a small workload'}
    logStage('COMPLETE', 'dataset discovery and sampling')
    saveEvidence()
except BaseException as error:
    summary['errors'].append(f'Selection: {type(error).__name__}: {error}')
    saveEvidence()
    raise



# 3 — Image Enhancement
Resize RGB images to **512×512 without cropping**, then apply baseline, gamma (`Lout = Lin**0.7` on normalized luminance), global histogram equalization, or CLAHE (clip limit 2; 8×8 tiles) on the CIELAB **L** channel. Convert back to RGB and use ImageNet normalization. These fixed parameters apply to **both reference and evaluation** images. No parameters are tuned on test labels. Masks receive no enhancement; exact foreground-coverage resizing is the approved edge-aligned evaluation transform.


In [ ]:
# These pure transformations produce new in-memory arrays; original files remain untouched.
def enhanceImage(rgb, condition):
    if condition == 'baseline':
        return rgb.copy()
    lab = cv2.cvtColor(rgb, cv2.COLOR_RGB2LAB)
    lightness = lab[:, :, 0]
    if condition == 'gamma':
        lightness = np.rint(255 * (lightness.astype(np.float32) / 255) ** enhancementParameters['gamma']).astype(np.uint8)
    elif condition == 'histogram_equalization':
        lightness = cv2.equalizeHist(lightness)
    elif condition == 'clahe':
        lightness = cv2.createCLAHE(clipLimit=enhancementParameters['claheClipLimit'],
                                   tileGridSize=tuple(enhancementParameters['claheTileGrid'])).apply(lightness)
    else:
        raise ValueError(f'Unknown enhancement: {condition}')
    lab[:, :, 0] = lightness
    return cv2.cvtColor(lab, cv2.COLOR_LAB2RGB)

logStage('START', 'enhancement setup')
normalizationMean = torch.tensor([0.485, 0.456, 0.406])[:, None, None]
normalizationStd = torch.tensor([0.229, 0.224, 0.225])[:, None, None]
logStage('COMPLETE', 'enhancement setup')
# Actual enhancement/features are included in reference-fitting and inference timings.


# 4 — PaDiM Feature Extraction
Frozen ImageNet ResNet18 layers 1–3 produce 448 concatenated channels. Bilinear alignment to **layer2-derived 64×64** and one seed-fixed **32-channel subset** bound memory and are identical for all categories, conditions, and workload modes. Fit one regularized multivariate Gaussian per spatial position, using only normal training features. Batch-Welford covariance accumulation avoids storing the full reference feature set; float64 covariance plus **0.01 I** stabilizes the small-sample fit. No supervised classifier is trained.
Sources: [PaDiM paper](https://arxiv.org/abs/2011.08785), [torchvision ResNet18](https://docs.pytorch.org/vision/stable/models/generated/torchvision.models.resnet18.html). The grid is derived from actual 512 input tensors; this bounded parameterization is not a claim to reproduce the paper's benchmark scores.


In [ ]:
def deriveGrid(model):
    with torch.inference_mode():
        x = torch.zeros((1, 3, 512, 512), device=device)
        x = model.maxpool(model.relu(model.bn1(model.conv1(x))))
        first = model.layer1(x)
        second = model.layer2(first)
        third = model.layer3(second)
    shapes = [list(layer.shape) for layer in [first, second, third]]
    if second.shape[-2] != second.shape[-1]:
        raise ValueError('Non-square feature grid for the fixed square input.')
    return int(second.shape[-1]), shapes

# Load one fixed backbone; no random-weight fallback if pretrained loading fails.
logStage('START', 'pretrained backbone loading')
try:
    backbone = resnet18(weights=ResNet18_Weights.IMAGENET1K_V1).eval().requires_grad_(False).to(device)
    featureGrid, layerShapes = deriveGrid(backbone)
    if featureGrid != 64:
        raise ValueError('Unexpected layer2 shape for the approved 512 input.')
    summary['model'].update(featureGrid=featureGrid, layerShapes=layerShapes)
    featureIndices = torch.randperm(448, generator=torch.Generator().manual_seed(randomSeed))[:featureDimension].to(device)
    summary['model']['featureIndices'] = featureIndices.cpu().tolist()
    logStage('COMPLETE', 'pretrained backbone loading')
    saveEvidence()
except BaseException as error:
    summary['errors'].append(f'Backbone: {type(error).__name__}: {error}')
    saveEvidence()
    raise


def extractFeatures(paths, condition):
    inputs = []
    for imagePath in paths:
        with Image.open(imagePath) as image:
            rgb = np.asarray(image.convert('RGB').resize((imageSize, imageSize), Image.Resampling.BILINEAR))
        tensor = torch.from_numpy(enhanceImage(rgb, condition)).permute(2, 0, 1).float() / 255
        inputs.append((tensor - normalizationMean) / normalizationStd)
    with torch.inference_mode():
        x = torch.stack(inputs).to(device)
        x = backbone.maxpool(backbone.relu(backbone.bn1(backbone.conv1(x))))
        layer1 = backbone.layer1(x)
        layer2 = backbone.layer2(layer1)
        layer3 = backbone.layer3(layer2)
        aligned = [F.interpolate(layer, size=(featureGrid, featureGrid), mode='bilinear', align_corners=False)
                   for layer in [layer1, layer2, layer3]]
        embeddings = torch.cat(aligned, dim=1)[:, featureIndices]
    result = embeddings.permute(0, 2, 3, 1).reshape(len(paths), -1, featureDimension).cpu().numpy().astype(np.float64)
    if result.shape != (len(paths), featureGrid ** 2, featureDimension) or not np.isfinite(result).all():
        raise ValueError('Unexpected/nonfinite backbone features; no automatic setting reduction.')
    return result

def fitPadim(references, category, condition):
    logStage('START', 'reference fitting (enhancement/features/covariance)', category, condition, 0, len(references))
    count = 0
    mean = np.zeros((featureGrid ** 2, featureDimension), dtype=np.float64)
    secondMoment = np.zeros((featureGrid ** 2, featureDimension, featureDimension), dtype=np.float64)
    for start in range(0, len(references), batchSize):
        features = extractFeatures([p for _, p in references[start:start+batchSize]], condition)
        newCount = len(features)
        batchMean = features.mean(axis=0)
        centered = features - batchMean
        delta = batchMean - mean
        # Merge batch and running centered sums; no evaluation images enter this fit.
        secondMoment += np.einsum('npd,npe->pde', centered, centered)
        secondMoment += np.einsum('pd,pe->pde', delta, delta) * count * newCount / (count + newCount)
        mean += delta * newCount / (count + newCount)
        count += newCount
        if count % 5 == 0:
            logStage('PROGRESS', 'reference fitting (enhancement/features/covariance)', category, condition, count, len(references))
    covariance = secondMoment / (count - 1) + covarianceRegularization * np.eye(featureDimension)
    if not np.isfinite(covariance).all():
        raise ValueError('Nonfinite spatial covariance.')
    precision = np.linalg.inv(covariance)
    if not np.isfinite(precision).all():
        raise ValueError('Nonfinite spatial precision.')
    logStage('COMPLETE', 'reference fitting (enhancement/features/covariance)', category, condition, count, len(references))
    return mean, precision


# 5 — Anomaly Scoring and Evaluation
Audit **every selected public anomaly mask before any model scoring**. Exact source-footprint coverage is copied from the tested Batch 3 implementation: a target cell is occupied when its edge-aligned source footprint overlaps a positive native pixel. Audit original alignment/binary values, native/target areas and 8-connected components, positive boxes, representative support for **every native component**, merging and native-equivalent area inflation. Invalid/missing masks or unrepresented components make category AU-PRO unavailable without excluding images; valid public labels still support AUROC. Normal labels imply all-background masks. Packed-bit caches keep the all-category audit small in memory.
Score by unchanged Mahalanobis distance, upsample to **512×512**, smooth with unchanged σ=4, and take the maximum for image AUROC. No per-image normalization. AU-PRO0.05 uses the unchanged exact equal-region weighted threshold sweep, FPR interval [0,0.05], normalized partial area, and 8-connected regions under the approved coverage semantics. No unsupported value is replaced with zero.
Coverage preserves occupancy, can merge regions and overstate area; square resizing distorts shape. These accepted effects create a **new, non-combinable protocol** relative to 256/NEAREST. Batch 3 fabric AUROC regressed. The fresh all-eight small pilot needs Sam review before medium. Model/memory failures stop with incomplete evidence; settings and sample sizes are never silently reduced.


In [ ]:
# Region-weighted ROC implements exact threshold sweeping with equal weight per defect region.
def auPro005(masks, scores):
    if masks.shape != scores.shape or masks.ndim != 3 or not np.isfinite(scores).all():
        return None, 'unavailable: invalid mask/score shapes or nonfinite scores'
    if not np.isin(masks, [0, 1]).all():
        return None, 'unavailable: masks must be binary'
    foreground = masks.astype(bool)
    if not foreground.any() or foreground.all():
        return None, 'unavailable: foreground regions and background pixels are required'
    regionWeights = np.ones(masks.shape, dtype=np.float64)
    for index, mask in enumerate(foreground):
        regions, regionCount = connectedComponents(mask, structure=np.ones((3, 3)))
        if regionCount:
            sizes = np.bincount(regions.ravel())
            regionWeights[index][mask] = 1.0 / sizes[regions[mask]]
    fpr, pro, _ = roc_curve(foreground.ravel(), scores.ravel(), sample_weight=regionWeights.ravel(), drop_intermediate=False)
    limit = 0.05
    inside = fpr < limit
    clippedFpr = np.r_[fpr[inside], limit]
    clippedPro = np.r_[pro[inside], np.interp(limit, fpr, pro)]
    return float(auc(clippedFpr, clippedPro) / limit), 'available'

def coverageMask(native, target=512):
    """Positive area in the exact edge-aligned target-cell footprint, not dilation.

    Source pixel cells [x,x+1) overlap target cell [j*W/T,(j+1)*W/T).
    floor/ceil integer bounds and a summed-area table test any positive overlap.
    A positive target cell can contain only a fraction of foreground: this is a
    conservative occupancy protocol, not an accurate area/region-shape estimate.
    """
    if native.ndim != 2 or not np.isin(native, [0, 1, 255]).all():
        raise ValueError('Expected a two-dimensional binary mask.')
    height, width = native.shape
    y0 = np.arange(target) * height // target
    y1 = (np.arange(1, target+1) * height + target-1) // target
    x0 = np.arange(target) * width // target
    x1 = (np.arange(1, target+1) * width + target-1) // target
    integral = np.pad(np.cumsum(np.cumsum(native > 0, axis=0, dtype=np.int64), axis=1), ((1, 0), (1, 0)))
    occupied = integral[y1[:, None], x1] - integral[y0[:, None], x1] - integral[y1[:, None], x0] + integral[y0[:, None], x0]
    return (occupied > 0).astype(np.uint8)

def maskStats(mask):
    binary = mask > 0
    regions, count = connectedComponents(binary, structure=np.ones((3, 3)))
    y, x = np.nonzero(binary)
    return dict(sizeWH=[mask.shape[1], mask.shape[0]], values=np.unique(mask).tolist(), area=int(binary.sum()),
                components=int(count), bboxXYXY=[int(x.min()), int(y.min()), int(x.max()+1), int(y.max()+1)] if len(x) else None), regions

def loadMasks(evaluations, category):
    masks = np.zeros((len(evaluations), imageSize, imageSize), dtype=np.uint8)
    reasons = []
    outcome = dict(category=category, selectedAnomalyCount=sum(g == 'bad' for g, _ in evaluations),
                   validMasks=0, masksWithGeometry=0, nativeComponents=0, targetComponents=0, representedNativeComponents=0, masksWithMerging=0)
    logStage('START', 'public mask alignment', category, 'all', 0, len(evaluations))
    for processed, (group, imagePath) in enumerate(evaluations, 1):
        if group == 'bad':
            maskPath = imagePath.parent.parent / 'ground_truth' / 'bad' / (imagePath.stem + '_mask.png')
            diagnostics = dict(category=category, relativeImagePath=imagePath.relative_to(datasetRoot).as_posix(),
                               relativeMaskPath=maskPath.relative_to(datasetRoot).as_posix(), resampling='exact-foreground-coverage-v1')
            try:
                with Image.open(imagePath) as image:
                    originalSize = image.size
                with Image.open(maskPath) as mask:
                    mask.load()
                    nativeMask = np.asarray(mask)
                    if mask.size != originalSize or nativeMask.ndim != 2 or not np.isin(nativeMask, [0, 255]).all() or not nativeMask.any():
                        raise ValueError('mask is empty, nonbinary, or inconsistent with the original image')
                    nativeStats, nativeRegions = maskStats(nativeMask)
                    resized = coverageMask(nativeMask, imageSize)
                    targetStats, _ = maskStats(resized)
                    # Project one actual positive pixel center per native component; merging is allowed.
                    ys, xs = np.nonzero(nativeMask)
                    _, first = np.unique(nativeRegions[ys, xs], return_index=True)
                    targetY = (2*ys[first]+1)*imageSize // (2*nativeMask.shape[0])
                    targetX = (2*xs[first]+1)*imageSize // (2*nativeMask.shape[1])
                    represented = int(np.count_nonzero(resized[targetY, targetX]))
                    width, height = originalSize
                    box = nativeStats['bboxXYXY']
                    expectedBox = [box[0]*imageSize//width, box[1]*imageSize//height,
                                   (box[2]*imageSize+width-1)//width, (box[3]*imageSize+height-1)//height]
                    diagnostics.update(imageSizeWH=list(originalSize), native=nativeStats, target=targetStats,
                                       representedNativeComponents=represented,
                                       areaInflationRatio=targetStats['area']*nativeMask.size/(imageSize**2*nativeStats['area']))
                    outcome['masksWithGeometry'] += 1
                    outcome['nativeComponents'] += nativeStats['components']
                    outcome['targetComponents'] += targetStats['components']
                    outcome['representedNativeComponents'] += represented
                    if represented != nativeStats['components'] or not targetStats['area']:
                        raise ValueError('Native anomaly component is unrepresented under coverage.')
                    if targetStats['bboxXYXY'] != expectedBox:
                        raise ValueError('Coverage mask does not match the edge-aligned image geometry.')
                    masks[processed-1] = resized
                    outcome['validMasks'] += 1
                    outcome['masksWithMerging'] += targetStats['components'] < nativeStats['components']
                    diagnostics['status'] = 'valid'
            except MemoryError:
                raise  # Resource failure stops the run; it is not an invalid annotation.
            except Exception as error:
                reason = f'{imagePath.name}: {type(error).__name__}: {error}'
                reasons.append(reason)
                diagnostics.update(status='invalid', reason=reason)
            summary['maskDiagnostics'].append(diagnostics)
        if processed % 5 == 0:
            logStage('PROGRESS', 'public mask alignment', category, 'all', processed, len(evaluations))
    outcome.update(status='unsupported' if reasons else 'pass', invalidMasks=len(reasons))
    summary['maskAuditOutcomes'].append(outcome)
    logStage('COMPLETE', 'public mask alignment', category, 'all', len(evaluations), len(evaluations))
    return masks, reasons

experimentsFinished = False
logStage('START', 'experiments')
try:
    packedMaskCache = {}
    for category in categories:
        auditedMasks, maskReasons = loadMasks(selectedData[category]['evaluation'], category)
        packedMaskCache[category] = (np.packbits(auditedMasks), auditedMasks.shape, maskReasons)
        summary['limitations'].extend(f'{category}: {reason}' for reason in maskReasons)
        del auditedMasks
        saveEvidence()
    summary['maskAuditComplete'] = True
    saveEvidence()
    for category in categories:
        references = selectedData[category]['reference']
        evaluations = selectedData[category]['evaluation']
        labels = np.array([int(group == 'bad') for group, _ in evaluations])
        bothClasses = set(labels.tolist()) == {0, 1}
        packed, maskShape, maskReasons = packedMaskCache.pop(category)
        masks = np.unpackbits(packed, count=int(np.prod(maskShape))).reshape(maskShape)
        del packed
        for condition in conditions:
            conditionStarted = time.perf_counter()
            logStage('START', 'condition', category, condition)
            imageValue, proValue = None, None
            imageStatus = proStatus = 'unavailable: at least two normal references and one evaluation image required'
            if len(references) >= 2 and evaluations:
                mean, precision = fitPadim(references, category, condition)
                scoreMaps = np.empty((len(evaluations), imageSize, imageSize), dtype=np.float32)
                logStage('START', 'inference (enhancement/features/scoring)', category, condition, 0, len(evaluations))
                for start in range(0, len(evaluations), batchSize):
                    batch = evaluations[start:start+batchSize]
                    difference = extractFeatures([p for _, p in batch], condition) - mean
                    distances = np.sqrt(np.maximum(0, np.einsum('npd,pde,npe->np', difference, precision, difference, optimize=True)))
                    maps = F.interpolate(torch.from_numpy(distances.reshape(-1, 1, featureGrid, featureGrid)),
                                         size=(imageSize, imageSize), mode='bilinear', align_corners=False).numpy()[:, 0]
                    for offset, scoreMap in enumerate(maps):
                        scoreMaps[start+offset] = gaussian_filter(scoreMap, sigma=4.0)
                    processed = start + len(batch)
                    if processed % 5 == 0:
                        logStage('PROGRESS', 'inference (enhancement/features/scoring)', category, condition, processed, len(evaluations))
                logStage('COMPLETE', 'inference (enhancement/features/scoring)', category, condition, len(evaluations), len(evaluations))
                logStage('START', 'metric evaluation', category, condition)
                if not np.isfinite(scoreMaps).all():
                    raise ValueError(f'{category}/{condition}: nonfinite anomaly scores')
                imageStatus = proStatus = 'unavailable: evaluation requires both normal and anomaly classes'
                if bothClasses:
                    imageValue = float(roc_auc_score(labels, scoreMaps.max(axis=(1, 2))))
                    imageStatus = 'available'
                    if maskReasons:
                        proStatus = 'unavailable: one or more selected public anomaly masks are missing or invalid; see summary limitations'
                    else:
                        proValue, proStatus = auPro005(masks, scoreMaps)
                del scoreMaps, mean, precision
                logStage('COMPLETE', 'metric evaluation', category, condition)
            for metricName, value, status in [('image_AUROC', imageValue, imageStatus), ('AU-PRO0.05', proValue, proStatus)]:
                metricRows.append(dict(dataset='MVTec AD 2', category=category, condition=condition, workloadMode=workloadSize,
                                       referenceCount=len(references), evaluationCount=len(evaluations),
                                       normalEvaluationCount=int((labels == 0).sum()), anomalyEvaluationCount=int((labels == 1).sum()),
                                       metricName=metricName, metricValue=value, metricStatus=status,
                                       runtimeSeconds=time.perf_counter()-conditionStarted))
                summary['metricAvailability'].append(dict(category=category, condition=condition, metric=metricName, status=status))
            if resource is not None:
                summary['memoryUsage']['cpuProcessLifetimePeakRssBytes'] = int(resource.getrusage(resource.RUSAGE_SELF).ru_maxrss * (1 if sys.platform == 'darwin' else 1024))
            if device.type == 'cuda':
                summary['memoryUsage']['gpuPeakAllocatedBytes'] = int(torch.cuda.max_memory_allocated(device))
            logStage('COMPLETE', 'condition', category, condition)
            saveEvidence()  # Preserve observed timings and completed metrics; run remains incomplete.
        del masks
    experimentsFinished = True
except BaseException as error:
    summary['errors'].append(f'Experiments: {type(error).__name__}: {error}')
    saveEvidence()
    raise
logStage('COMPLETE', 'experiments')


# 6 — Results and Evidence
Only after all selected category/condition configurations (32 by default) have been visited is execution marked **complete**. Unsupported metrics stay blank with reasons; complete execution is not a claim that every metric was available or that enhancement helped. `metrics.csv`, `experiment_summary.json`, and `sample_manifest.csv` remain in the mode-specific UTC run folder. Conditions share identities, model, feature subset, and evaluation policy. A disconnect leaves the current run incomplete and preserves prior run folders; restart creates a new run rather than overwriting evidence. Model checkpoints/resume remain outside scope. The summary records protocol/version, frozen identity comparison, file manifest, mask audit outcomes, geometry, timings, and measured memory where available.
Carlos executes on the real datasets; Sam reviews the evidence; Carlos accepts. Synthetic implementation tests establish no real research result or statistical significance.
This all-eight small run is exploratory screening. Do not choose final categories or declare enhancement superior from this run alone; Team review determines any follow-up scope.
Archived 256 values are a separate series. Team approval of this protocol is not acceptance of the new real all-eight results; Carlos runs Colab and supplies the new timestamped evidence folder to Sam before medium.


In [ ]:
# Finalize only if scoring/evaluation reached the explicit end-of-run gate.
logStage('START', 'results')
if not summary['maskAuditComplete'] or not experimentsFinished or len(metricRows) != len(categories) * len(conditions) * 2:
    saveEvidence()
    raise RuntimeError('Incomplete experiments: refusing to mark the run complete.')

for row in metricRows:
    value = '' if row['metricValue'] is None else f"{row['metricValue']:.4f}"
    print(f"{row['category']}/{row['condition']} | {row['metricName']}={value} | {row['metricStatus']}", flush=True)
logStage('COMPLETE', 'results')
saveEvidence()  # All computation persisted; still incomplete until the final notebook gate.
logStage('COMPLETE', 'notebook')
summary['status'] = 'complete'
summary['totalRuntimeSeconds'] = time.perf_counter() - runStarted
temporaryPath = runRoot / 'experiment_summary.json.tmp'
temporaryPath.write_text(json.dumps(summary, indent=2, allow_nan=False), encoding='utf-8')
temporaryPath.replace(runRoot / 'experiment_summary.json')
print('Evidence:', runRoot, flush=True)
